# Cats vs Dogs  🐱 🐶

In [1]:
import os
import cv2
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Activation, Conv2D, MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam,SGD,Adagrad,Adadelta,RMSprop

import matplotlib.pyplot as plt
from tensorflow.keras.layers import Input

from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

Modelin sınıflandıracağı kedi ve köpek türlerini tanımlayıp veri setindeki görsellerin bulunduğu klasör yolunu belirliyoruz.

In [2]:
labels=['Cat', 'Dog']
img_path='train/'

Tüm sınıf klasörlerini dolaşıp görsellerin tam yollarını ve karşılık gelen etiketlerini listelere ekliyoruz.

In [3]:
img_list = []  # tüm resim dosyalarının tam yolunu saklamak için boş bir liste oluşturur
label_list = []  # her resme karşılık gelen etiketleri saklamak için boş bir liste oluşturur

for label in labels:  # labels listesindeki her bir sınıf/etiket için döngü başlatır
    for img_file in os.listdir(img_path + label):  # ilgili etiket klasörünün içindeki tüm dosyaları listeler
        img_list.append(img_path + label + "/" + img_file)  # resmin tam dosya yolunu oluşturup img_list listesine ekler
        label_list.append(label)  # bu resme ait etiketi label_list listesine ekler

Görsel yolları ve etiketleri bir araya getirerek bir DataFrame oluşturuyoruz.

In [4]:
df=pd.DataFrame({'img':img_list,'label':label_list})

In [5]:
df

,img,label
0,train/Cat/cat.0.jpg,Cat
1,train/Cat/cat.1.jpg,Cat
2,train/Cat/cat.10.jpg,Cat
3,train/Cat/cat.100.jpg,Cat
4,train/Cat/cat.1000.jpg,Cat
...,...,...
24995,train/Dog/dog.9995.jpg,Dog
24996,train/Dog/dog.9996.jpg,Dog
24997,train/Dog/dog.9997.jpg,Dog
24998,train/Dog/dog.9998.jpg,Dog


Köpek ve Kedi türlerini sayısal değerlere dönüştürerek modelin anlayabileceği şekilde etiketleri encode ediyoruz.

In [6]:
d={'Cat':0, 'Dog':1} 
df['label_encoded']=df['label'].map(d)

In [7]:
df

,img,label,label_encoded
0,train/Cat/cat.0.jpg,Cat,0
1,train/Cat/cat.1.jpg,Cat,0
2,train/Cat/cat.10.jpg,Cat,0
3,train/Cat/cat.100.jpg,Cat,0
4,train/Cat/cat.1000.jpg,Cat,0
...,...,...,...
24995,train/Dog/dog.9995.jpg,Dog,1
24996,train/Dog/dog.9996.jpg,Dog,1
24997,train/Dog/dog.9997.jpg,Dog,1
24998,train/Dog/dog.9998.jpg,Dog,1


Görselleri okuyup aynı boyuta getirerek normalize ediyoruz ve model eğitimi için bir liste içinde topluyoruz.

In [8]:
x = []  # islenmis resimleri tutmak icin bos bir liste olusturur
for img in df['img']:  # dataframe icindeki img sutununda yer alan her bir resim yolu icin donguye girer
    img = cv2.imread(str(img))  # verilen dosya yolundaki resmi okur ve numpy array haline getirir
    img = cv2.resize(img, (150, 150))  # tum resimleri ayni boyuta getirmek icin yeniden boyutlandirir
    img = img / 255.0  # piksel degerlerini 0-1 araligina cekerek normalize eder
    x.append(img)  # islenmis resmi x listesine ekler

In [9]:
x=np.array(x)

In [10]:
x.shape

(25000, 150, 150, 3)

In [11]:
y=df[['label_encoded']]

In [12]:
x_train,x_test,y_train,y_test=train_test_split(x,y, random_state=42, test_size=0.20)

y_train ve y_test i modelde kullanılabilir hale getirmek için tam sayı formatına çeviriyoruz.

In [13]:
y_train=np.array(y_train, dtype=np.int32)
y_test=np.array(y_test, dtype=np.int32)

CNN modelimizi oluşturuyoruz.

In [14]:
model = Sequential()
model.add(Input(shape=(150, 150, 3)))

model.add(Conv2D(32, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

# Modeli derliyoruz
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

Ve modeli eğitiyoruz.

In [15]:
history=model.fit(x_train,y_train,validation_data=(x_test,y_test), epochs=20,verbose=1)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 211s 334ms/step - accuracy: 0.6251 - loss: 0.6774 - val_accuracy: 0.7032 - val_loss: 0.5974
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 224s 358ms/step - accuracy: 0.7225 - loss: 0.5482 - val_accuracy: 0.7344 - val_loss: 0.5187
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 228s 304ms/step - accuracy: 0.7657 - loss: 0.4836 - val_accuracy: 0.7646 - val_loss: 0.4868
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 200s 301ms/step - accuracy: 0.8037 - loss: 0.4253 - val_accuracy: 0.7834 - val_loss: 0.4670
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 188s 302ms/step - accuracy: 0.8284 - loss: 0.3739 - val_accuracy: 0.7610 - val_loss: 0.4927
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 229s 344ms/step - accuracy: 0.8583 - loss: 0.3260 - val_accuracy: 0.7836 - val_loss: 0.5017
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 194s 311ms/step - accuracy: 0.8815 - loss: 0.2793 - val_accuracy: 0.7916 - val_loss: 0.4726
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 190s 304ms/step - accuracy: 0.9007 -

In [16]:
model.save('CatOrDog_model.h5')

tesekkürler :)